# Notebook 09-COLAB — GridSearchCV con Validación Cruzada Estratificada (K=5)
### Entorno: Google Colab — Runtime T4 GPU

---

## Objetivo
Ejecutar **GridSearchCV** con **StratifiedKFold (K=5)** de forma **individual** para:
- **SVM** — `LinearSVC` (scikit-learn, CPU optimizado)
- **Regresión Logística** — `LogisticRegression` (scikit-learn, CPU optimizado)

Sobre el corpus completo de entrenamiento (`train_val_set.csv`), registrando para cada modelo:
- **F1-Score Macro** — mejor resultado del GridSearch
- **Tiempo Total de Entrenamiento** — en minutos
- **Pico Máximo de RAM** — medido con `memory_profiler`, en GB

> **Nota de entorno:** `LinearSVC` y `LogisticRegression` son estimadores basados en CPU.
> La T4 aporta 15 GB de RAM de sistema adicional que acelera el procesamiento paralelo
> (`n_jobs=-1`) durante el GridSearch. El uso de GPU se reserva para etapas Transformer.

---

## Celda 0 — Verificación del Entorno T4 y Recursos Disponibles

In [ ]:
import subprocess
import platform
import os

print("=" * 60)
print("  VERIFICACION DEL ENTORNO COLAB")
print("=" * 60)

# Informacion de GPU
try:
    resultado_gpu = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True
    )
    if resultado_gpu.returncode == 0:
        lineas = resultado_gpu.stdout.strip().split('\n')
        for linea in lineas:
            partes = [p.strip() for p in linea.split(',')]
            print(f"  GPU detectada  : {partes[0]}")
            print(f"  VRAM total     : {partes[1]}")
            print(f"  Driver         : {partes[2]}")
    else:
        print("  GPU            : No detectada (verificar runtime T4)")
except FileNotFoundError:
    print("  GPU            : nvidia-smi no disponible")

# RAM del sistema
try:
    import psutil
    ram_total_gb = psutil.virtual_memory().total / (1024 ** 3)
    ram_disp_gb  = psutil.virtual_memory().available / (1024 ** 3)
    print(f"  RAM total      : {ram_total_gb:.1f} GB")
    print(f"  RAM disponible : {ram_disp_gb:.1f} GB")
except ImportError:
    pass

# CPUs disponibles
print(f"  CPUs logicas   : {os.cpu_count()}")
print(f"  Python         : {platform.python_version()}")
print("=" * 60)

## Celda 1 — Instalación de Dependencias

> Instala `memory_profiler` e `imbalanced-learn`. El resto ya viene preinstalado en Colab.

In [ ]:
# Instalar dependencias que no vienen por defecto en Colab
!pip install memory_profiler imbalanced-learn -q

import memory_profiler
import imblearn
print(f"memory_profiler : {memory_profiler.__version__}")
print(f"imbalanced-learn: {imblearn.__version__}")
print("Dependencias instaladas correctamente.")

## Celda 2 — Montaje de Google Drive y Carga del Dataset

Ajusta `DRIVE_DATA_PATH` a la carpeta de tu Drive donde subiste `train_val_set.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# --- CONFIGURACION: ajusta esta ruta a tu Drive ---
DRIVE_DATA_PATH = '/content/drive/MyDrive/ProyectoFinal_ML/data/train_val_set.csv'
# --------------------------------------------------

import pandas as pd
import os

if not os.path.exists(DRIVE_DATA_PATH):
    raise FileNotFoundError(
        f"No se encontro el archivo en:\n  {DRIVE_DATA_PATH}\n"
        "Verifica que subiste 'train_val_set.csv' a la ruta indicada."
    )

df = pd.read_csv(DRIVE_DATA_PATH, sep=';')
df['texto_lineal'] = df['texto_lineal'].fillna('')

X = df[['texto_lineal']]
y = df['label']

print("Dataset cargado exitosamente.")
print(f"  Total de registros : {len(df):,}")
print(f"  Distribucion de clases:\n{y.value_counts().to_string()}")

## Celda 3 — Importaciones Generales

In [ ]:
import time
import warnings
import gc

from memory_profiler import memory_usage

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.base import clone

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
print("Librerias cargadas correctamente.")

## Celda 4 — Configuración del Pipeline Base y StratifiedKFold (K=5)

In [ ]:
# Validacion Cruzada Estratificada: garantiza proporciones de clase en cada fold
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Submuestreo aleatorio para balancear clases dentro de cada fold
undersampler = RandomUnderSampler(random_state=42)

# Vectorizador TF-IDF sobre la columna de texto preprocesado
preprocessor = ColumnTransformer(
    transformers=[('tfidf', TfidfVectorizer(), 'texto_lineal')]
)

# Pipeline molde — el clasificador se sustituye dinamicamente por cada malla
pipeline_base = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer',   preprocessor),
    ('classifier',   LogisticRegression())  # placeholder
])

print("Pipeline base configurado:")
print("  Submuestreo -> TF-IDF -> Clasificador")
print(f"  StratifiedKFold(K=5, shuffle=True, random_state=42)")

## Celda 5 — Helper: Entrenamiento con Medición de RAM y Tiempo

`memory_profiler.memory_usage()` ejecuta el `fit()` monitoreando el proceso cada
`interval=0.1 s` e incluye procesos hijos (workers de `n_jobs=-1`).
El pico en MB se convierte a GB para el reporte final.

In [ ]:
def entrenar_con_metricas(grid_search_obj, X_data, y_data, nombre_modelo):
    """
    Ejecuta grid_search_obj.fit(X_data, y_data) midiendo:
      - Pico de RAM del proceso + workers (memory_profiler) en GB
      - Tiempo total en minutos
    Retorna: (best_score, tiempo_min, ram_gb)
    """
    separador = '=' * 60
    print(f"\n{separador}")
    print(f"  Entrenando: {nombre_modelo}")
    print(f"  GridSearchCV K=5 | corpus completo | n_jobs=-1")
    print(f"{separador}")

    def _fit():
        grid_search_obj.fit(X_data, y_data)

    t_inicio = time.time()

    muestras_ram = memory_usage(
        (_fit, [], {}),
        interval=0.1,
        include_children=True,
        max_usage=False,
        retval=False
    )

    t_fin = time.time()

    pico_ram_mb = max(muestras_ram)
    pico_ram_gb = pico_ram_mb / 1024
    tiempo_min  = (t_fin - t_inicio) / 60
    best_score  = grid_search_obj.best_score_

    print(f"\n  Entrenamiento finalizado.")
    print(f"  Mejor configuracion: {grid_search_obj.best_params_}")
    return best_score, tiempo_min, pico_ram_gb

## Celda 6 — GridSearchCV para SVM (LinearSVC)

In [ ]:
param_grid_svm = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LinearSVC(max_iter=2000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

grid_svm = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_svm,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_svm, tiempo_svm, ram_svm = entrenar_con_metricas(
    grid_svm, X, y, "SVM — LinearSVC"
)

gc.collect()
print("\nMemoria liberada. Listo para Regresion Logistica.")

## Celda 7 — GridSearchCV para Regresión Logística

In [ ]:
param_grid_log = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

grid_log = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_log,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_log, tiempo_log, ram_log = entrenar_con_metricas(
    grid_log, X, y, "Regresion Logistica"
)

gc.collect()
print("\nMemoria liberada.")

## Celda 8 — Tabla de Resultados Finales

In [ ]:
print("\n" + "=" * 65)
print("  RESULTADOS FINALES — GridSearchCV K=5 (Corpus Completo)")
print("  Entorno: Google Colab — T4 GPU Runtime")
print("=" * 65)
print(f"  SVM:            F1-Score: {f1_svm:.2f} | Tiempo: {tiempo_svm:.1f} min | RAM: {ram_svm:.2f} GB")
print(f"  Reg. Logistica: F1-Score: {f1_log:.2f} | Tiempo: {tiempo_log:.1f} min | RAM: {ram_log:.2f} GB")
print("=" * 65)

resultados = pd.DataFrame({
    'Modelo':          ['SVM (LinearSVC)', 'Regresion Logistica'],
    'F1-Score Macro':  [round(f1_svm, 4),  round(f1_log, 4)],
    'Tiempo (min)':    [round(tiempo_svm, 2), round(tiempo_log, 2)],
    'RAM Pico (GB)':   [round(ram_svm, 3),  round(ram_log, 3)],
    'Mejor C':         [
        grid_svm.best_params_.get('classifier__C', 'N/A'),
        grid_log.best_params_.get('classifier__C', 'N/A')
    ],
    'max_features':    [
        grid_svm.best_params_.get('vectorizer__tfidf__max_features', 'N/A'),
        grid_log.best_params_.get('vectorizer__tfidf__max_features', 'N/A')
    ]
})

display(resultados.set_index('Modelo'))

## Celda 9 — Exportar Modelos y Resultados a Google Drive

In [ ]:
import joblib

# Carpeta de salida en Drive
DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/ProyectoFinal_ML/data/'
os.makedirs(DRIVE_OUTPUT_PATH, exist_ok=True)

# Guardar el mejor modelo SVM
path_svm = os.path.join(DRIVE_OUTPUT_PATH, 'best_svm_pipeline_colab.pkl')
joblib.dump(grid_svm.best_estimator_, path_svm)
print(f"Modelo SVM exportado    : {path_svm}")

# Guardar el mejor modelo Logistico
path_log = os.path.join(DRIVE_OUTPUT_PATH, 'best_log_pipeline_colab.pkl')
joblib.dump(grid_log.best_estimator_, path_log)
print(f"Modelo Logistico exportado: {path_log}")

# Guardar tabla de resultados en CSV
path_csv = os.path.join(DRIVE_OUTPUT_PATH, 'resultados_gridsearch_colab.csv')
resultados.to_csv(path_csv, index=False, sep=';')
print(f"Tabla de resultados guardada: {path_csv}")

print("\nTodos los artefactos exportados a Google Drive correctamente.")